<center>

# CITS5504: Project 2
## Semester 1
### Nisari Ramachandran
### 23873626

</center>

# ETL process for creating the nodes and Relationships from the Original Dataset

In [1]:
#libraries
import subprocess
import sys

for pkg in ["pandas","requests"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m","pip","install",pkg])

import pandas as pd
import requests

In [2]:
#download the dataset from the url 
url = "https://618598184-files.gitbook.io/~/files/v0/b/gitbook-x-prod.appspot.com/o/spaces%2FDoBTcHHBV80kLIWxmuWv%2Fuploads%2FtENZqqLI6Gq8KHuaRbvl%2FProject2_Dataset_Corrected.csv?alt=media&token=24ceee2f-7372-4b7f-a78e-d075ec59dea0"
filename = "Project2_Dataset_Corrected.csv"

with open(filename,"wb") as f:
    f.write(requests.get(url).content)
    
#load the dataset    
og_data = pd.read_csv(filename)

In [3]:
#basic dataset cleaning
og_data.columns = og_data.columns.str.strip().str.lower().str.replace(' ', '_')

og_data = og_data.rename(
    
    columns={"national_remoteness_areas": "remoteness",
             "sa4_name_2021": "sa4",
             "national_lga_name_2024": "lga"}
    )

og_data.head(3)

,id,crash_id,state,month,year,dayweek,time,crash_type,number_fatalities,bus_involvement,...,age,remoteness,sa4,lga,national_road_type,christmas_period,easter_period,age_group,day_of_week,time_of_day
0,1,20241115,NSW,12,2024,Friday,4:00,Single,1,No,...,74,Inner Regional Australia,Riverina,Wagga Wagga,Arterial Road,Yes,No,65_to_74,Weekday,Night
1,2,20241125,NSW,12,2024,Friday,6:15,Single,1,No,...,19,Inner Regional Australia,Sydney - Baulkham Hills and Hawkesbury,Hawkesbury,Local Road,No,No,17_to_25,Weekday,Day
2,3,20246013,TAS,12,2024,Friday,9:43,Single,1,No,...,33,Inner Regional Australia,Launceston and North East,Northern Midlands,Local Road,Yes,No,26_to_39,Weekday,Day


In [4]:
#column names in the dataset
og_data.columns

Index(['id', 'crash_id', 'state', 'month', 'year', 'dayweek', 'time',
       'crash_type', 'number_fatalities', 'bus_involvement',
       'heavy_rigid_truck_involvement', 'articulated_truck_involvement',
       'speed_limit', 'road_user', 'gender', 'age', 'remoteness', 'sa4', 'lga',
       'national_road_type', 'christmas_period', 'easter_period', 'age_group',
       'day_of_week', 'time_of_day'],
      dtype='object')

## Nodes

In [5]:
og_data["road_user_id"] = og_data.groupby(["crash_id", "road_user", "gender", "age"]).cumcount()
#create additional property(like key) so that each combination of the crash_id and road_user is preserved
og_data["ru_uid"] = og_data["crash_id"].astype(str) + '_' + og_data["road_user"] + '_' + og_data["gender"] + '_' + og_data["age"].astype(str) + '_' + og_data["road_user_id"].astype(str)


In [6]:
#creating nodes
og_data[["crash_id","crash_type","number_fatalities"]].drop_duplicates().to_csv("crash.csv",index=False)

og_data[["ru_uid","road_user","gender","age","age_group"]]\
    .drop_duplicates().to_csv("road_user.csv",index=False)
    
    
    
    
    
og_data[["bus_involvement","heavy_rigid_truck_involvement","articulated_truck_involvement"]].drop_duplicates().to_csv("vehicles.csv",index=False)
og_data[["speed_limit","national_road_type"]].drop_duplicates().to_csv("road_details.csv",index=False)
og_data[["year","month","dayweek","day_of_week","christmas_period","easter_period"]].drop_duplicates().to_csv("date.csv",index=False)
og_data[["time","time_of_day"]].drop_duplicates().to_csv("time.csv",index=False)
og_data[["state"]].drop_duplicates().to_csv("state.csv",index=False)
og_data[["remoteness"]].drop_duplicates().to_csv("remoteness.csv",index=False)
og_data[["sa4"]].drop_duplicates().to_csv("sa4.csv",index=False)
og_data[["lga"]].drop_duplicates().to_csv("lga.csv",index=False)



## Relationships

In [7]:
#creating relationships
#crash - road user
og_data[["crash_id","ru_uid"]].drop_duplicates().to_csv("crash_user.csv",index=False)

#state - sa4
og_data[["sa4", "state"]].drop_duplicates().to_csv("sa4_state.csv", index=False)

#rest of them there would be multiple for the same crash if there has been multiple fatalities,
#so we should remove the duplicates to remove redundancy
og_data[["crash_id","time","time_of_day"]].drop_duplicates().to_csv("crash_time.csv",index=False)
og_data[["crash_id","state"]].drop_duplicates().to_csv("crash_state.csv",index=False)

og_data[["crash_id","speed_limit","national_road_type"]].drop_duplicates().to_csv("crash_road.csv",index=False)
og_data[["crash_id","year","month","dayweek","day_of_week","christmas_period","easter_period"]].drop_duplicates().to_csv("crash_date.csv",index=False)
og_data[["crash_id","bus_involvement","heavy_rigid_truck_involvement","articulated_truck_involvement"]].drop_duplicates().to_csv("crash_vehicle.csv",index=False)
og_data[["crash_id","remoteness"]].drop_duplicates().to_csv("crash_remoteness.csv",index=False)
og_data[["crash_id","sa4"]].drop_duplicates().to_csv("crash_sa4.csv",index=False)
og_data[["crash_id","lga"]].drop_duplicates().to_csv("crash_lga.csv",index=False)




In [8]:
#sa4 state check whether there is any sa4 with multiple states
sa4_state_check = og_data.groupby("sa4")["state"].nunique().reset_index()
sa4_state_check = sa4_state_check[sa4_state_check["state"] > 1]

# Show problematic SA4s
print(sa4_state_check)

Empty DataFrame
Columns: [sa4, state]
Index: []


In [9]:
og_data[["crash_id", "time"]].duplicated().sum()


807

In [10]:
df = pd.read_csv("crash_time.csv")
print(df.duplicated().sum()) 

0
